<a href="https://colab.research.google.com/github/KrasnovaOV/Big-Data-Analytics/blob/main/%D0%A1%D0%B1%D0%BE%D1%80_%D0%B8_%D1%80%D0%B0%D0%B1%D0%BE%D1%82%D1%8B_%D1%81_%D0%B4%D0%B0%D0%BD%D0%BD%D1%8B%D0%BC%D0%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Задание**

Вопросы по заданию

Домашнее задание к занятию "Культура сбора и работы с данными"
Возьмите свою схему хранилища данных из домашнего задания №3 и предложите 4–5 проверок на качество данных.

Пример проверки:
«В справочнике ‘клиенты’ у каждого человека должен быть заполнен email-адрес. Это должен быть валидный адрес, уникальный в рамках системы».

In [1]:
import sqlite3
import pandas as pd
import re
from IPython.display import display

# 1. Создаём базу данных SQLite в памяти
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

# 2. Создаём таблицы модели «звезда»
cursor.executescript("""
CREATE TABLE dim_date (
    date_key INTEGER PRIMARY KEY,
    full_date DATE,
    day_of_week TEXT,
    month INTEGER,
    year INTEGER,
    is_weekend BOOLEAN
);

CREATE TABLE dim_product (
    product_key INTEGER PRIMARY KEY,
    product_id TEXT,
    product_name TEXT,
    category TEXT,
    brand TEXT,
    price_segment TEXT,
    is_active BOOLEAN
);

CREATE TABLE dim_customer (
    customer_key INTEGER PRIMARY KEY,
    customer_id TEXT,
    customer_name TEXT,
    email TEXT,
    city TEXT,
    customer_segment TEXT,
    loyalty_status TEXT,
    registration_date DATE
);

CREATE TABLE dim_store (
    store_key INTEGER PRIMARY KEY,
    store_id TEXT,
    store_name TEXT,
    store_type TEXT,
    city TEXT,
    is_active BOOLEAN
);

CREATE TABLE fact_sales (
    sale_id INTEGER PRIMARY KEY,
    order_id INTEGER,
    date_key INTEGER,
    product_key INTEGER,
    customer_key INTEGER,
    store_key INTEGER,
    quantity INTEGER,
    unit_price DECIMAL(12,2),
    discount_amount DECIMAL(12,2),
    revenue DECIMAL(14,2),
    cost_amount DECIMAL(14,2),
    gross_profit DECIMAL(14,2),
    promotion_flag BOOLEAN
);
""")

# 3. Добавляем тестовые данные
cursor.executemany(
    """
    INSERT INTO dim_date VALUES (?, ?, ?, ?, ?, ?)
    """,
    [
        (20261120, "2026-11-20", "пятница", 11, 2026, 0),
        (20261121, "2026-11-21", "суббота", 11, 2026, 1)
    ]
)

cursor.executemany(
    """
    INSERT INTO dim_product VALUES (?, ?, ?, ?, ?, ?, ?)
    """,
    [
        (1, "P-001", "Футболка", "Одежда", "Brand A", "Эконом", 1),
        (2, "P-002", "Смартфон", "Электроника", "Brand B", "Премиум", 1)
    ]
)

cursor.executemany(
    """
    INSERT INTO dim_customer VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """,
    [
        (1, "C-001", "Иван Иванов", "ivan@example.com", "Москва", "Постоянный", "Активен", "2024-01-10"),
        (2, "C-002", "Мария Петрова", "maria@example", "Санкт-Петербург", "Новый", "Не активен", "2026-11-01"),
        (3, "C-003", "Пётр Сидоров", "ivan@example.com", "Казань", "Постоянный", "Активен", "2023-06-15")
    ]
)

cursor.executemany(
    """
    INSERT INTO dim_store VALUES (?, ?, ?, ?, ?, ?)
    """,
    [
        (1, "S-001", "Интернет-магазин", "Онлайн", "Москва", 1),
        (2, "S-002", "Магазин Тверская", "Офлайн", "Москва", 1)
    ]
)

cursor.executemany(
    """
    INSERT INTO fact_sales VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """,
    [
        # Корректная запись
        (1, 1001, 20261120, 1, 1, 1, 2, 2000.00, 400.00, 3200.00, 1200.00, 2000.00, 1),

        # Отрицательное количество
        (2, 1002, 20261121, 2, 2, 2, -1, 50000.00, 0.00, -50000.00, 30000.00, -80000.00, 0),

        # Неверная выручка
        (3, 1003, 20261121, 1, 3, 1, 1, 2000.00, 0.00, 5000.00, 1200.00, 3800.00, 0),

        # Несуществующий клиент
        (4, 1004, 20261120, 2, 999, 1, 1, 50000.00, 5000.00, 45000.00, 30000.00, 15000.00, 0),

        # Дубликат продажи
        (5, 1001, 20261120, 1, 1, 1, 2, 2000.00, 400.00, 3200.00, 1200.00, 2000.00, 1)
    ]
)

conn.commit()

# 4. Загружаем таблицы в pandas для проверок
fact_sales = pd.read_sql_query("SELECT * FROM fact_sales", conn)
dim_customer = pd.read_sql_query("SELECT * FROM dim_customer", conn)
dim_product = pd.read_sql_query("SELECT * FROM dim_product", conn)
dim_date = pd.read_sql_query("SELECT * FROM dim_date", conn)
dim_store = pd.read_sql_query("SELECT * FROM dim_store", conn)

# 5. Функция для оформления результатов проверки
def show_check(title, errors):
    print(f"\n=== {title} ===")

    if errors.empty:
        print("Ошибок не найдено.")
    else:
        print(f"Найдено ошибок: {len(errors)}")
        display(errors)

# Проверка 1. Email клиента: заполнен, валиден и уникален
email_pattern = r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"

invalid_email = dim_customer[
    dim_customer["email"].isna()
    | (dim_customer["email"].astype(str).str.strip() == "")
    | ~dim_customer["email"].astype(str).str.match(email_pattern)
]

duplicate_email = dim_customer[
    dim_customer["email"].duplicated(keep=False)
    & dim_customer["email"].notna()
]

email_errors = pd.concat([invalid_email, duplicate_email]).drop_duplicates()

show_check(
    "Проверка 1. Email клиента заполнен, валиден и уникален",
    email_errors[["customer_key", "customer_id", "customer_name", "email"]]
)

# Проверка 2. Количество товара должно быть положительным
negative_quantity = fact_sales[fact_sales["quantity"] <= 0]

show_check(
    "Проверка 2. Количество товара больше нуля",
    negative_quantity[["sale_id", "order_id", "product_key", "quantity"]]
)

# Проверка 3. Финансовые показатели корректны
fact_sales["expected_revenue"] = (
    fact_sales["unit_price"] - fact_sales["discount_amount"]
) * fact_sales["quantity"]

financial_errors = fact_sales[
    (fact_sales["unit_price"] < 0)
    | (fact_sales["discount_amount"] < 0)
    | (fact_sales["cost_amount"] < 0)
    | (fact_sales["revenue"] < 0)
    | (fact_sales["gross_profit"] < 0)
    | (fact_sales["revenue"] != fact_sales["expected_revenue"])
]

show_check(
    "Проверка 3. Цена, скидка, себестоимость и выручка корректны",
    financial_errors[
        [
            "sale_id",
            "order_id",
            "quantity",
            "unit_price",
            "discount_amount",
            "revenue",
            "expected_revenue",
            "gross_profit"
        ]
    ]
)

# Проверка 4. Все внешние ключи существуют в измерениях
invalid_product = fact_sales[
    ~fact_sales["product_key"].isin(dim_product["product_key"])
]

invalid_customer = fact_sales[
    ~fact_sales["customer_key"].isin(dim_customer["customer_key"])
]

invalid_store = fact_sales[
    ~fact_sales["store_key"].isin(dim_store["store_key"])
]

invalid_date = fact_sales[
    ~fact_sales["date_key"].isin(dim_date["date_key"])
]

referential_errors = pd.concat([
    invalid_product.assign(error_type="Неизвестный товар"),
    invalid_customer.assign(error_type="Неизвестный клиент"),
    invalid_store.assign(error_type="Неизвестный магазин"),
    invalid_date.assign(error_type="Неизвестная дата")
])

show_check(
    "Проверка 4. Ссылочная целостность фактов и измерений",
    referential_errors[
        [
            "sale_id",
            "order_id",
            "date_key",
            "product_key",
            "customer_key",
            "store_key",
            "error_type"
        ]
    ]
)

# Проверка 5. Дубликаты строк фактов
duplicate_sales = fact_sales[
    fact_sales.duplicated(
        subset=["order_id", "date_key", "product_key", "store_key"],
        keep=False
    )
]

show_check(
    "Проверка 5. Дубликаты товарных позиций в заказах",
    duplicate_sales[
        [
            "sale_id",
            "order_id",
            "date_key",
            "product_key",
            "store_key",
            "quantity",
            "revenue"
        ]
    ]
)

# 6. Итоговый отчёт
quality_report = pd.DataFrame(
    {
        "Проверка": [
            "Email клиента заполнен, валиден и уникален",
            "Количество товара больше нуля",
            "Финансовые показатели корректны",
            "Ссылочная целостность фактов и измерений",
            "Дубликаты товарных позиций в заказах"
        ],
        "Количество ошибок": [
            len(email_errors),
            len(negative_quantity),
            len(financial_errors),
            len(referential_errors),
            len(duplicate_sales)
        ]
    }
)

print("\n=== Итоговый отчёт о качестве данных ===")
display(quality_report)

conn.close()


=== Проверка 1. Email клиента заполнен, валиден и уникален ===
Найдено ошибок: 3


,customer_key,customer_id,customer_name,email
1,2,C-002,Мария Петрова,maria@example
0,1,C-001,Иван Иванов,ivan@example.com
2,3,C-003,Пётр Сидоров,ivan@example.com



=== Проверка 2. Количество товара больше нуля ===
Найдено ошибок: 1


,sale_id,order_id,product_key,quantity
1,2,1002,2,-1



=== Проверка 3. Цена, скидка, себестоимость и выручка корректны ===
Найдено ошибок: 2


,sale_id,order_id,quantity,unit_price,discount_amount,revenue,expected_revenue,gross_profit
1,2,1002,-1,50000,0,-50000,-50000,-80000
2,3,1003,1,2000,0,5000,2000,3800


... (2 outputs left out)


=== Проверка 5. Дубликаты товарных позиций в заказах ===
Найдено ошибок: 2


,sale_id,order_id,date_key,product_key,store_key,quantity,revenue
0,1,1001,20261120,1,1,2,3200
4,5,1001,20261120,1,1,2,3200



=== Итоговый отчёт о качестве данных ===


,Проверка,Количество ошибок
0,"Email клиента заполнен, валиден и уникален",3
1,Количество товара больше нуля,1
2,Финансовые показатели корректны,2
3,Ссылочная целостность фактов и измерений,1
4,Дубликаты товарных позиций в заказах,2


**Приведение данных в нужный формат**

In [2]:
import re
import pandas as pd
from IPython.display import display

# Сохраняем исходные данные для отчёта «до исправления»
fact_sales_before = fact_sales.copy()
dim_customer_before = dim_customer.copy()

# Счётчик исправлений
corrections_log = []

# ============================================
# ИСПРАВЛЕНИЕ 1. Email клиентов
# ============================================
email_pattern = r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"

# Заменяем пустые и невалидные email
invalid_email_mask = (
    dim_customer["email"].isna()
    | (dim_customer["email"].astype(str).str.strip() == "")
    | ~dim_customer["email"].astype(str).str.match(email_pattern)
)

invalid_email_count = invalid_email_mask.sum()

dim_customer.loc[invalid_email_mask, "email"] = (
    "unknown_" + dim_customer.loc[invalid_email_mask, "customer_id"].astype(str)
    + "@example.com"
)

if invalid_email_count > 0:
    corrections_log.append(
        f"Исправлено невалидных или пустых email: {invalid_email_count}"
    )

# Убираем дубли email: оставляем первое вхождение
duplicate_email_mask = dim_customer["email"].duplicated(keep="first")
duplicate_email_count = duplicate_email_mask.sum()

for index in dim_customer[duplicate_email_mask].index:
    dim_customer.loc[index, "email"] = (
        "unknown_"
        + str(dim_customer.loc[index, "customer_id"])
        + "_"
        + str(index)
        + "@example.com"
    )

if duplicate_email_count > 0:
    corrections_log.append(
        f"Заменено дублирующихся email: {duplicate_email_count}"
    )

# ============================================
# ИСПРАВЛЕНИЕ 2. Отрицательное количество товара
# ============================================
negative_quantity_mask = fact_sales["quantity"] <= 0
negative_quantity_count = negative_quantity_mask.sum()

fact_sales = fact_sales[~negative_quantity_mask].copy()

if negative_quantity_count > 0:
    corrections_log.append(
        f"Удалено записей с неположительным количеством товара: {negative_quantity_count}"
    )

# ============================================
# ИСПРАВЛЕНИЕ 3. Финансовые показатели
# ============================================
fact_sales["revenue"] = (
    (fact_sales["unit_price"] - fact_sales["discount_amount"])
    * fact_sales["quantity"]
).round(2)

fact_sales["gross_profit"] = (
    fact_sales["revenue"] - fact_sales["cost_amount"]
).round(2)

financial_fixed_count = len(
    fact_sales_before[
        (fact_sales_before["revenue"] < 0)
        | (fact_sales_before["gross_profit"] < 0)
    ]
)

if financial_fixed_count > 0:
    corrections_log.append(
        f"Пересчитаны финансовые показатели в {financial_fixed_count} записях"
    )

# ============================================
# ИСПРАВЛЕНИЕ 4. Ссылочная целостность
# ============================================
invalid_customer_mask = ~fact_sales["customer_key"].isin(
    dim_customer["customer_key"]
)

invalid_customer_count = invalid_customer_mask.sum()
fact_sales = fact_sales[~invalid_customer_mask].copy()

if invalid_customer_count > 0:
    corrections_log.append(
        f"Удалено записей с несуществующим клиентом: {invalid_customer_count}"
    )

# ============================================
# ИСПРАВЛЕНИЕ 5. Дубликаты позиций заказов
# ============================================
duplicate_mask = fact_sales.duplicated(
    subset=["order_id", "date_key", "product_key", "store_key"],
    keep="first"
)

duplicate_count = duplicate_mask.sum()
fact_sales = fact_sales[~duplicate_mask].copy()

if duplicate_count > 0:
    corrections_log.append(
        f"Удалено дублирующихся записей продаж: {duplicate_count}"
    )

# ============================================
# ПОВТОРНЫЕ ПРОВЕРКИ ПОСЛЕ ИСПРАВЛЕНИЙ
# ============================================

# Проверка 1. Email
invalid_email_after = dim_customer[
    dim_customer["email"].isna()
    | (dim_customer["email"].astype(str).str.strip() == "")
    | ~dim_customer["email"].astype(str).str.match(email_pattern)
    | dim_customer["email"].duplicated(keep=False)
]

# Проверка 2. Количество
negative_quantity_after = fact_sales[fact_sales["quantity"] <= 0]

# Проверка 3. Финансы
fact_sales["expected_revenue"] = (
    (fact_sales["unit_price"] - fact_sales["discount_amount"])
    * fact_sales["quantity"]
)

financial_errors_after = fact_sales[
    (fact_sales["unit_price"] < 0)
    | (fact_sales["discount_amount"] < 0)
    | (fact_sales["cost_amount"] < 0)
    | (fact_sales["revenue"] < 0)
    | (fact_sales["gross_profit"] < 0)
    | (fact_sales["revenue"] != fact_sales["expected_revenue"])
]

# Проверка 4. Ссылочная целостность
referential_errors_after = pd.concat([
    fact_sales[~fact_sales["product_key"].isin(dim_product["product_key"])],
    fact_sales[~fact_sales["customer_key"].isin(dim_customer["customer_key"])],
    fact_sales[~fact_sales["store_key"].isin(dim_store["store_key"])],
    fact_sales[~fact_sales["date_key"].isin(dim_date["date_key"])]
]).drop_duplicates(subset=["sale_id"])

# Проверка 5. Дубликаты
duplicate_sales_after = fact_sales[
    fact_sales.duplicated(
        subset=["order_id", "date_key", "product_key", "store_key"],
        keep=False
    )
]

# ============================================
# ИТОГОВЫЙ ОТЧЁТ
# ============================================
final_report = pd.DataFrame(
    {
        "Проверка": [
            "Email клиента заполнен, валиден и уникален",
            "Количество товара больше нуля",
            "Цена, скидка, себестоимость и выручка корректны",
            "Ссылочная целостность фактов и измерений",
            "Дубликаты товарных позиций в заказах"
        ],
        "Ошибок до исправления": [
            3,
            1,
            2,
            1,
            2
        ],
        "Ошибок после исправления": [
            len(invalid_email_after),
            len(negative_quantity_after),
            len(financial_errors_after),
            len(referential_errors_after),
            len(duplicate_sales_after)
        ]
    }
)

final_report["Статус"] = final_report["Ошибок после исправления"].apply(
    lambda x: "Проверка пройдена" if x == 0 else "Требует внимания"
)

print("=== Журнал выполненных исправлений ===")
for item in corrections_log:
    print("-", item)

print("\n=== Итоговый отчёт о качестве данных после исправлений ===")
display(final_report)

print("\n=== Исправленная таблица клиентов ===")
display(dim_customer)

print("\n=== Исправленная таблица фактов продаж ===")
display(
    fact_sales[
        [
            "sale_id",
            "order_id",
            "product_key",
            "customer_key",
            "store_key",
            "quantity",
            "unit_price",
            "discount_amount",
            "revenue",
            "cost_amount",
            "gross_profit"
        ]
    ]
)

=== Журнал выполненных исправлений ===
- Исправлено невалидных или пустых email: 1
- Заменено дублирующихся email: 1
- Удалено записей с неположительным количеством товара: 1
- Пересчитаны финансовые показатели в 1 записях
- Удалено записей с несуществующим клиентом: 1
- Удалено дублирующихся записей продаж: 1

=== Итоговый отчёт о качестве данных после исправлений ===


,Проверка,Ошибок до исправления,Ошибок после исправления,Статус
0,"Email клиента заполнен, валиден и уникален",3,0,Проверка пройдена
1,Количество товара больше нуля,1,0,Проверка пройдена
2,"Цена, скидка, себестоимость и выручка корректны",2,0,Проверка пройдена
3,Ссылочная целостность фактов и измерений,1,0,Проверка пройдена
4,Дубликаты товарных позиций в заказах,2,0,Проверка пройдена



=== Исправленная таблица клиентов ===


,customer_key,customer_id,customer_name,email,city,customer_segment,loyalty_status,registration_date
0,1,C-001,Иван Иванов,ivan@example.com,Москва,Постоянный,Активен,2024-01-10
1,2,C-002,Мария Петрова,unknown_C-002@example.com,Санкт-Петербург,Новый,Не активен,2026-11-01
2,3,C-003,Пётр Сидоров,unknown_C-003_2@example.com,Казань,Постоянный,Активен,2023-06-15



=== Исправленная таблица фактов продаж ===


,sale_id,order_id,product_key,customer_key,store_key,quantity,unit_price,discount_amount,revenue,cost_amount,gross_profit
0,1,1001,1,1,1,2,2000,400,3200,1200,2000
2,3,1003,1,3,1,1,2000,0,2000,1200,800
